# Track B: Predicting Risky Choices
**IML Midterm Project, CS-3410 / MAT-3211-1, Ashoka University**  
Name: *Kanchan Bharti* | Kaggle username: *kanchanbharti039*

**Note:** the data is *simulated*.

## Research question
Do framing (gain/loss frame) and features based on expected value help predict risky choices for **new participants**, compared with using only the problem numbers?

## What I expect (hypotheses)
1. **Exp 1:** Adding frame/domain and expected-value features will improve AUC. Person details (age, numeracy, risk score) will not help much for new people.
2. **Exp 2:** A random split will give a higher (too optimistic) AUC than a split by participant, especially if the model can recognise a person.
3. **Exp 3:** `response_time_ms` is recorded *with* the choice, so I leave it out of my model. I expect it adds nothing anyway.

## Plan
- Metric: **ROC-AUC** (also log-loss and accuracy at 0.5). The model outputs a score for "chose risky"; ROC-AUC is the chance that a random risky decision gets a higher score than a random safe one (0.5 = guessing, 1 = perfect).
- Validation: `GroupKFold` on `participant_id` (5 shuffled folds, same folds for every model), because the test set has people not seen in training.
- Drop `realised_payout_inr` (leakage). Only classical ML (no neural networks).
- AI declaration: *Used Claude*

**How to run:** set `folder` in the next code cell to your existing folder (it must contain `train.csv`, `test.csv`, `sample_submission.csv`), then *Run All*. All result files are saved **into that same folder** (no new folder is created).

## 1. Settings and data

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from scipy.stats import rankdata
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold, KFold
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
warnings.filterwarnings("ignore")

# ===== THE ONLY PLACE TO SET THE FOLDER =====
# Existing folder with train.csv, test.csv, sample_submission.csv. All results are saved here too.
folder = r"C:\Users\Lenvo\Downloads\iml-2026-track-b-risky-choice"

SEED, K = 0, 5                                   # seed and number of participant folds (used everywhere)
np.random.seed(SEED)
path = lambda name: os.path.join(folder, name)   # file inside the folder
rank = lambda p: rankdata(p) / len(p)

train = pd.read_csv(path("train.csv"))
test = pd.read_csv(path("test.csv"))
sample = pd.read_csv(path("sample_submission.csv"))
print("pandas", pd.__version__, "| numpy", np.__version__, "| sklearn", sklearn.__version__)
print(train.shape, test.shape)
train.head()

## 2. Check the data and remove leakage

In [ ]:
info = {"n_train": len(train), "n_test": len(test),
        "n_train_participants": int(train.participant_id.nunique()),
        "n_test_participants": int(test.participant_id.nunique()),
        "participants_in_both": len(set(train.participant_id) & set(test.participant_id)),
        "share_risky": float(train.chose_risky.mean()),
        "paid_share": float((train.session_type == "paid").mean()),
        "payout_missing": int(train.realised_payout_inr.isna().sum())}
print("Columns only in train:", set(train.columns) - set(test.columns))
print(info)

# realised_payout_inr is paid AFTER the choice and is not in the test file -> leakage, drop it
train = train.drop(columns=["realised_payout_inr"])

- `realised_payout_inr` is missing for 4 600 of 8 400 rows, is only known after the choice and is not in the test file, so it is leakage. I drop it.
- Train and test have **no participant in common** (210 vs 90), so I must validate by participant.
- The classes are balanced (about 48% risky), so AUC is a fine metric.

## 3. Make new features
Simple formulas on the problem numbers (nothing is learned from the data), then the column groups used in the experiments.

In [ ]:
def add_features(df):
    df = df.copy()
    df["ev_diff"] = df.prob_high * df.risky_high_inr + (1 - df.prob_high) * df.risky_low_inr - df.sure_amount_inr
    df["spread"] = df.risky_high_inr - df.risky_low_inr
    df["ev_ratio"] = df.ev_diff / df.spread                       # EV difference relative to the stakes
    df["sure_pos"] = (df.sure_amount_inr - df.risky_low_inr) / df.spread
    df["small_p"] = (df.prob_high <= 0.10).astype(int)            # people overweight small probabilities
    df["large_p"] = (df.prob_high >= 0.90).astype(int)
    df["cell"] = df.problem_domain + "_" + df.frame
    df["ev_end"] = df.ev_diff / (df.endowment_inr + 1)            # EV difference on the scale of the endowment

    def pt_value(x, a, lam):      # prospect-theory value function (Tversky & Kahneman, 1992)
        return np.where(x >= 0, np.abs(x) ** a, -lam * np.abs(x) ** a)
    def pt_weight(p, g):          # probability weighting (small p overweighted)
        return p ** g / (p ** g + (1 - p) ** g) ** (1 / g)

    # value difference for 4 typical parameter sets (a, lam, g); fixed formulas, uses outcomes AS SHOWN (frame included)
    for a, lam, g in [(0.88, 2.25, 0.61), (0.7, 1.5, 0.7), (0.88, 1.0, 0.8), (0.6, 2.5, 0.6)]:
        df[f"pt_{a}_{lam}_{g}"] = (pt_weight(df.prob_high, g) * pt_value(df.risky_high_inr, a, lam)
                                   + pt_weight(1 - df.prob_high, g) * pt_value(df.risky_low_inr, a, lam)
                                   - pt_value(df.sure_amount_inr, a, lam)) / (df.endowment_inr + 1)
    df["paid"] = (df.session_type == "paid").astype(float)
    pt_cols = [k for k in df.columns if k.startswith("pt_")]
    for c in ["gain_gain_frame", "gain_loss_frame", "loss_none", "mixed_none"]:   # separate effect per domain/frame cell
        df["ev_ratio_" + c] = df.ev_ratio * (df.cell == c)
        for p in pt_cols:
            df[p + "_" + c] = df[p] * (df.cell == c)
    for c in ["self_reported_risk_taking", "numeracy_score", "age_years", "time_pressure", "paid"]:   # EV x person traits
        df["evr_x_" + c] = df.ev_ratio * df[c]
        df["eve_x_" + c] = df.ev_end * df[c]
    return df

train, test = add_features(train), add_features(test)
assert np.isfinite(train.select_dtypes("number").values).all()
y, groups = train.chose_risky, train.participant_id
train["pid"] = train.participant_id            # only used in Experiment 2

nominal = ["prob_high", "risky_high_inr", "risky_low_inr", "sure_amount_inr"]
derived = ["ev_diff", "ev_ratio", "sure_pos", "spread", "small_p", "large_p"]
derived_new = derived + ["ev_end"] + [c for c in train.columns if c.startswith("pt_") and c.count("_") == 3]
frame_cols = ["problem_domain", "frame", "cell"]
person_cols = ["time_pressure", "numeracy_score", "age_years", "self_reported_risk_taking"]
interactions_old = [c for c in train.columns if c.startswith("ev_ratio_")]
interactions = [c for c in train.columns if c.startswith(("ev_ratio_", "evr_x_", "eve_x_"))
                or (c.startswith("pt_") and c.count("_") > 3)]
C_FINAL = 0.1        # L2 strength of the final logistic regression (the sweep in section 8 shows 0.003 to 1 are the same)

lin_num, tree_num = nominal + derived_new + interactions, nominal + derived_new + person_cols + ["paid", "trial_index"]
lin_cols, tree_cols = lin_num + frame_cols, tree_num + frame_cols
print("domain/frame cells: share choosing risky"); print(train.groupby("cell").chose_risky.mean().round(3))

The same gain-domain problems are chosen risky more often in the **loss frame** (0.56) than in the gain frame (0.40). The expected-value rule cannot see this, so frame should help. (Only the gain domain has both frames, so frame and domain cannot be separated elsewhere.)

## 4. Evaluation function
`run()` is used for **every** model: the same 5 shuffled `GroupKFold` splits (a participant's 40 decisions always stay together), AUC / log-loss / accuracy per fold, and one row added to the results table (columns of the required `experimental_results.csv`). Scaling and one-hot encoding are inside the pipeline, so they are fitted on training folds only.

In [ ]:
rows, oof_store = [], {}      # one row per run; out-of-fold scores and per-fold AUCs of every run

def make_model(clf, num_cols, cat_cols=()):
    tr = [("num", StandardScaler(), list(num_cols))]
    if cat_cols:
        tr.append(("cat", OneHotEncoder(handle_unknown="ignore"), list(cat_cols)))
    return make_pipeline(ColumnTransformer(tr), clf)

def make_lr(C=C_FINAL):                      # final logistic regression
    return make_model(LogisticRegression(C=C, max_iter=5000), lin_num, frame_cols)

def make_et():                               # ExtraTrees (strongly regularised: leaves of at least 30 decisions)
    prep = ColumnTransformer([("num", "passthrough", tree_num),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), frame_cols)])
    return make_pipeline(prep, ExtraTreesClassifier(n_estimators=300, min_samples_leaf=30, max_features=0.6,
                                                    n_jobs=-1, random_state=SEED))

def splits(scheme):
    if scheme == "group":
        return GroupKFold(n_splits=K, shuffle=True, random_state=SEED).split(train, y, groups)
    return KFold(n_splits=K, shuffle=True, random_state=SEED).split(train, y)

def fit_model(model_fn, cols, kind="proba"):
    """returns a function (train_idx, val_idx) -> (scores, kind) for run()"""
    def fp(tr, va):
        m = model_fn().fit(train.iloc[tr][cols], y.iloc[tr])
        s = m.predict_proba(train.iloc[va][cols])[:, 1] if kind == "proba" else m.decision_function(train.iloc[va][cols])
        return s, kind
    return fp

def run(exp, name, model, fset, hyper, fit_predict, scheme="group", notes=""):
    t0 = time.time()
    auc, ll, acc, oof = [], [], [], np.zeros(len(train))
    for tr, va in splits(scheme):
        s, kind = fit_predict(tr, va)
        yt = y.iloc[va]
        auc.append(roc_auc_score(yt, s))
        ll.append(log_loss(yt, np.clip(s, 1e-6, 1 - 1e-6)) if kind == "proba" else np.nan)
        acc.append(accuracy_score(yt, s > (0.5 if kind == "proba" else 0)) if kind in ("proba", "rule") else np.nan)
        oof[va] = s
    stat = lambda v: (np.mean(v), np.std(v, ddof=1))
    (am, asd), (lm, lsd), (cm, csd) = stat(auc), stat(ll), stat(acc)
    ok = lambda v: round(v, 4) if not np.isnan(v) else ""
    rows.append(dict(run_id=f"B-{len(rows)+1:02d}", track="B", experiment=exp, model=model, feature_set=fset,
                     validation_scheme="group_kfold_participant_shuffled" if scheme == "group" else "kfold_random_shuffled",
                     n_folds=K, seed=SEED, hyperparameters=hyper, primary_metric_name="roc_auc",
                     primary_metric_mean=round(am, 4), primary_metric_sd=round(asd, 4),
                     secondary_metric_name="log_loss", secondary_metric_mean=ok(lm), secondary_metric_sd=ok(lsd),
                     threshold={"proba": "0.5", "rule": "0 (EV>0)"}.get(kind, ""), runtime_s=round(time.time() - t0, 1),
                     code_reference=f"IML_Mid_term_Project_Track_B_improved.ipynb ({exp})",
                     notes=((f"accuracy={cm:.4f}+-{csd:.4f}; " if not np.isnan(cm) else "") + notes).strip("; ")))
    oof_store[name] = (oof, auc)
    print(f"{name:46s} AUC {am:.4f} +- {asd:.4f} | log-loss {lm:.3f} | accuracy {cm:.3f}")
    return oof

## 5. Baselines
A constant predictor, the expected-value rule (no training), and four lecture models on the 4 raw problem columns only.

In [ ]:
run("baseline", "constant", "DummyClassifier(prior)", "nominal(4)", "strategy=prior",
    fit_model(lambda: DummyClassifier(strategy="prior"), nominal))
run("baseline", "ev_rule", "expected-value rule (no training)", "ev_diff", "score=ev_diff",
    lambda tr, va: (train.ev_diff.iloc[va].values, "rule"))

for name, clf, kind in [("LogisticRegression", LogisticRegression(max_iter=1000), "proba"),
                        ("LinearDiscriminantAnalysis", LinearDiscriminantAnalysis(), "proba"),
                        ("GaussianNB", GaussianNB(), "proba"),
                        ("LinearSVC", LinearSVC(max_iter=10000), "score")]:
    run("baseline", name + " (raw 4)", name, "nominal(4)", "default",
        fit_model(lambda clf=clf: make_model(clf, nominal), nominal, kind))

**Observation:** the expected-value rule (0.728) is as good as any model that only sees the 4 raw numbers (about 0.72). Expected value is probability x outcome, which a linear model on the raw columns cannot build by itself, so **features matter more than model type**.

## 6. Experiment 1: which features help? (logistic regression)
Only the feature set changes: (a) raw numbers, (b) + EV features, (c) + domain/frame, (d) + person/session info, (e) + frame x EV interactions, (f) + prospect-theory features and trait interactions (final).

In [ ]:
exp1 = [("1a nominal only",                         nominal,                                     [],                          1.0,     "nominal(4)"),
        ("1b + EV features",                        nominal + derived,                           [],                          1.0,     "+ derived(10)"),
        ("1c + domain and frame",                   nominal + derived,                           frame_cols,                  1.0,     "+ domain/frame(13)"),
        ("1d + person/context",                     nominal + derived + person_cols,             frame_cols + ["session_type"], 1.0,   "+ person, session(18)"),
        ("1e + frame x EV interactions",            nominal + derived + interactions_old,        frame_cols,                  1.0,     "1c + 4 ev_ratio x cell"),
        ("1f + prospect-theory + trait interactions", lin_num,                                   frame_cols,                  C_FINAL, "lin_cols (48)")]
for name, num, cat, C_, fset in exp1:
    oof = run("exp1_features", name, "LogisticRegression", fset, f"C={C_}",
              fit_model(lambda num=num, cat=cat, C_=C_: make_model(LogisticRegression(C=C_, max_iter=5000), num, cat), num + cat))
oof_lr = oof                      # out-of-fold scores of the final logistic regression (1f)

**Result:** EV features give the biggest jump (0.72 to 0.785), domain/frame add more (0.814). Person/context info does **not** help (0.812). Interactions and prospect-theory features add only about 0.002 each, which is within fold noise (s.d. about 0.003). This matches my hypothesis: framing matters, age/numeracy/risk score do not help for unseen people. Model 1f is the final logistic regression.

## 7. Experiments 2 and 3: split scheme and response time
**Exp 2:** same model, only the split changes (random `KFold` vs participant `GroupKFold`), also with a one-hot `participant_id` so the model can "remember" a person. **Exp 3:** add `response_time_ms` (not known before the choice, so not allowed in the final model).

In [ ]:
lr_pid = lambda: make_model(LogisticRegression(C=C_FINAL, max_iter=5000), lin_num, frame_cols + ["pid"])
for name, fn, cols, scheme, fset in [("2 random KFold",                     make_lr, lin_cols,          "random", "lin_cols"),
                                     ("2 participant GroupKFold",           make_lr, lin_cols,          "group",  "lin_cols"),
                                     ("2 + participant_id, random KFold",   lr_pid,  lin_cols + ["pid"], "random", "lin_cols+participant_id"),
                                     ("2 + participant_id, GroupKFold",     lr_pid,  lin_cols + ["pid"], "group",  "lin_cols+participant_id")]:
    run("exp2_split", name, "LogisticRegression (final LR)", fset, f"C={C_FINAL}", fit_model(fn, cols), scheme)

run("exp3_response_time", "3 + response_time_ms", "LogisticRegression (final LR)", "lin_cols+response_time_ms", f"C={C_FINAL}",
    fit_model(lambda: make_model(LogisticRegression(C=C_FINAL, max_iter=10000), lin_num + ["response_time_ms"], frame_cols),
              lin_cols + ["response_time_ms"]),
    notes="response_time_ms is measured with the choice -> NOT allowed in the final model")
print(train.groupby("chose_risky").response_time_ms.mean().round(0))

**Exp 2:** for the final model the random and participant splits give the same AUC (0.818), so my hypothesis was only partly right: the random split is *not* optimistic here. With `participant_id` the random split rises to 0.839 while the participant split stays at 0.818; the paired gap (+0.021 +- 0.022 over 5 folds) points the expected way but is noisy. The test set has new people, so the participant split is the honest one.

**Exp 3:** response time changes AUC by -0.0001 and the mean time is almost equal for both choices (807 vs 823 ms), so leaving it out loses nothing.

## 8. Regularisation, ExtraTrees and the final blend
1. **Regularisation sweep** of the logistic regression (C = 0.003, 0.03, 1; C = 0.1 is the final model above).
2. **ExtraTrees**: trees can find interactions and also use `trial_index` and the person columns (`trial_index` is known before the choice, so it is allowed).
3. **Rank-average blend** of logistic regression and ExtraTrees. AUC only depends on ranking, so averaging ranks is a safe way to combine models.

In [ ]:
for C_ in [0.003, 0.03, 1.0]:
    run("tuning", f"LR C={C_}", "LogisticRegression", "lin_cols", f"C={C_}", fit_model(lambda C_=C_: make_lr(C_), lin_cols))

oof_et = run("model_comparison", "ExtraTrees", "ExtraTreesClassifier", "tree_cols (24)",
             "n_estimators=300,min_samples_leaf=30,max_features=0.6", fit_model(make_et, tree_cols))

def blend(tr, va):
    a = make_lr().fit(train.iloc[tr][lin_cols], y.iloc[tr]).predict_proba(train.iloc[va][lin_cols])[:, 1]
    b = make_et().fit(train.iloc[tr][tree_cols], y.iloc[tr]).predict_proba(train.iloc[va][tree_cols])[:, 1]
    return (rank(a) + rank(b)) / 2, "score"

oof_bl = run("final", "FINAL blend (LR + ExtraTrees rank average)", "rank average of final LR and ExtraTrees",
             "lin_cols + tree_cols", f"LR C={C_FINAL}; ET as above", blend,
             notes="submitted model; ranks computed within each fold, scores are not calibrated")

**Result:** C from 0.003 to 1 changes nothing (AUC 0.8175 to 0.8182). ExtraTrees alone is about the same as the logistic regression (0.819). The blend gives **0.820**, a gain of about +0.002 over the logistic regression; it is positive in 4 of 5 folds but close to the noise level, so I treat it as a small, not certain, improvement.

## 9. Paired differences between runs
Because all runs use the same folds, I can subtract the per-fold AUCs (mean +- s.d. over the 5 folds) to see which differences are real.

In [ ]:
def paired(a, b):
    d = np.array(oof_store[a][1]) - np.array(oof_store[b][1])
    return float(d.mean()), float(d.std(ddof=1))

F = "1f + prospect-theory + trait interactions"
paired_diffs = {"final LR - EV rule": paired(F, "ev_rule"),
                "final LR - raw-number LR": paired(F, "1a nominal only"),
                "domain/frame added (1c - 1b)": paired("1c + domain and frame", "1b + EV features"),
                "person traits added (1d - 1c)": paired("1d + person/context", "1c + domain and frame"),
                "prospect-theory added (1f - 1e)": paired(F, "1e + frame x EV interactions"),
                "ExtraTrees - final LR": paired("ExtraTrees", F),
                "blend - final LR": paired("FINAL blend (LR + ExtraTrees rank average)", F),
                "random - group split": paired("2 random KFold", "2 participant GroupKFold"),
                "random - group split, with participant_id": paired("2 + participant_id, random KFold", "2 + participant_id, GroupKFold"),
                "with response time - final LR": paired("3 + response_time_ms", F)}
for k, (m, s) in paired_diffs.items():
    print(f"{k:45s} {m:+.4f} +- {s:.4f}")

## 10. Figures, calibration and error analysis
The figure (saved into the folder) shows the framing effect, ROC curves from pooled out-of-fold scores, and the calibration of the logistic regression. The error analysis looks at AUC per domain/frame cell, per session type and accuracy per participant.

In [ ]:
# calibration of the final logistic regression (10 bins, pooled out-of-fold)
bins = pd.cut(oof_lr, bins=np.linspace(0, 1, 11), include_lowest=True)
cal = pd.DataFrame({"pred": oof_lr, "act": y}).groupby(bins, observed=True).agg(pred=("pred", "mean"), act=("act", "mean"), n=("act", "size"))

plt.rcParams.update({"font.size": 9})
fig, ax = plt.subplots(1, 3, figsize=(10.5, 3.1))
cs = train.groupby("cell").chose_risky.mean()
ax[0].bar(range(4), cs.values, color="#4C72B0"); ax[0].set_xticks(range(4))
ax[0].set_xticklabels([s.replace("_", "\n", 1) for s in cs.index], fontsize=7)
ax[0].set_ylabel("share choosing risky"); ax[0].set_title("(a) Risky choices by domain/frame"); ax[0].set_ylim(0, 0.7)
for lab, sc, col in [("final blend", oof_bl, "#C44E52"), ("final LR", oof_lr, "#4C72B0"), ("EV rule", train.ev_diff.values, "#55A868")]:
    fp_, tp_, _ = roc_curve(y, sc); ax[1].plot(fp_, tp_, color=col, label=f"{lab} ({roc_auc_score(y, sc):.3f})")
ax[1].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].set_xlabel("false positive rate"); ax[1].set_ylabel("true positive rate")
ax[1].set_title("(b) ROC, pooled out-of-fold"); ax[1].legend(fontsize=7, loc="lower right")
ax[2].plot([0, 1], [0, 1], "k--", lw=.8); ax[2].plot(cal.pred, cal.act, "o-", color="#4C72B0")
ax[2].set_xlabel("predicted probability (LR)"); ax[2].set_ylabel("observed share risky"); ax[2].set_title("(c) Calibration of LR")
plt.tight_layout(); plt.savefig(path("fig_overview.png"), dpi=200); plt.show()

# error analysis
cell_stats = {c: dict(n=len(g), share_risky=float(y[g.index].mean()),
                      auc_ev=float(roc_auc_score(y[g.index], g.ev_diff)),
                      auc_lr=float(roc_auc_score(y[g.index], oof_lr[g.index])),
                      auc_blend=float(roc_auc_score(y[g.index], oof_bl[g.index]))) for c, g in train.groupby("cell")}
print(pd.DataFrame(cell_stats).T.round(3))
session_auc = {s: float(roc_auc_score(y[train.session_type == s], oof_lr[(train.session_type == s).values])) for s in ["paid", "hypothetical"]}
acc_person = pd.DataFrame({"p": groups, "ok": (oof_lr > 0.5) == y}).groupby("p").ok.mean()
print("AUC by session (LR):", {k: round(v, 3) for k, v in session_auc.items()})
print("accuracy per participant: min %.2f | mean %.2f | max %.2f | s.d. %.2f" % (acc_person.min(), acc_person.mean(), acc_person.max(), acc_person.std()))
print("calibration: mean absolute gap = %.4f" % ((cal.n * (cal.pred - cal.act).abs()).sum() / cal.n.sum()))

plt.figure(figsize=(4.8, 3.0)); plt.hist(acc_person.values, bins=20, color="#4C72B0")
plt.xlabel("out-of-fold accuracy of one participant"); plt.ylabel("participants"); plt.title("Accuracy per participant (final LR)")
plt.tight_layout(); plt.savefig(path("fig_participant_acc.png"), dpi=200); plt.show()

- The model beats the EV rule in every cell. The hardest cell is gain problems in the gain frame (AUC about 0.68); the easiest is mixed gambles (about 0.90).
- Calibration is good (mean gap about 0.007). Paid sessions are easier (AUC 0.85) than hypothetical ones (0.79).
- Accuracy depends a lot on the person (about 0.43 to 0.95): the model cannot know if a new person is a strong risk taker or very careful.

## 11. Final model, submission file and saving the results
The final model is fitted **once on all training data** and predicts the test file once. Because it is a rank average, the submitted `chose_risky` values are scores between 0 and 1 with the right ordering, which is all ROC-AUC needs (not calibrated probabilities). The last lines save `submission.csv`, `experimental_results.csv` and `report_numbers.json` into `folder`.

In [ ]:
lr_final = make_lr().fit(train[lin_cols], y)
et_final = make_et().fit(train[tree_cols], y)
p_test = (rank(lr_final.predict_proba(test[lin_cols])[:, 1]) + rank(et_final.predict_proba(test[tree_cols])[:, 1])) / 2

submission = pd.DataFrame({"decision_id": test.decision_id, "chose_risky": p_test})
submission = submission.set_index("decision_id").loc[sample.decision_id].reset_index()   # same row order as sample_submission
assert list(submission.columns) == list(sample.columns) and len(submission) == len(sample)
assert set(submission.decision_id) == set(sample.decision_id) and submission.chose_risky.notna().all()

submission.to_csv(path("submission.csv"), index=False)
pd.DataFrame(rows).to_csv(path("experimental_results.csv"), index=False)
json.dump(dict(data=info, paired_auc_differences=paired_diffs, per_cell=cell_stats, auc_by_session_LR=session_auc,
               acc_per_participant=dict(min=float(acc_person.min()), mean=float(acc_person.mean()), max=float(acc_person.max()),
                                        sd=float(acc_person.std())),
               n_features=dict(lin_cols=len(lin_cols), tree_cols=len(tree_cols)), folds=K, seed=SEED),
          open(path("report_numbers.json"), "w"), indent=1)
print("Saved in", folder, ":", ["submission.csv", "experimental_results.csv", "report_numbers.json", "fig_overview.png", "fig_participant_acc.png"])
submission.head()

## 12. Conclusion and limitations
**Answer:** for new participants, logistic regression with expected-value features, domain/frame and interactions reaches AUC about **0.82** (0.818 +- 0.013), compared with 0.73 for the expected-value rule and about 0.72 for models that only see the raw numbers. Blending it with ExtraTrees (no neural network) gives 0.820 +- 0.011. Framing helps; age, numeracy and risk score do not help for unseen people. A random split gave the same AUC for the simple model, but becomes optimistic if the model can learn who the person is.

**Limitations**
- The blend gain (about +0.002) is close to the CV noise level.
- The data is **simulated**, so this says nothing about real people; choices are random in the simulator, so a perfect score is impossible.
- Only 210 training participants and 5 folds (the s.d. is over folds), so differences below about 0.003 should not be over-interpreted.
- I chose the final model with the same CV, so the estimate may be slightly optimistic.
- I show associations only, not causes.